# Task 4 — Applications & Shortlisting (Explainability)
**Focus:** Add match explainability — matches include a structured explanation payload.

This is a **standalone notebook**. It re-loads students.csv / jobs.csv / matches.csv fresh
and re-defines the small set of helper functions needed, so it runs independently of
the Task 1+2+3 notebook.

## Cell 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import json
print('Libraries loaded')

Libraries loaded


## Cell 2 — Load data

In [2]:
students = pd.read_csv('../datasets/students.csv')
jobs     = pd.read_csv('../datasets/jobs.csv')
matches  = pd.read_csv('../datasets/matches.csv')

print('Students:', students.shape, '| Jobs:', jobs.shape, '| Matches:', matches.shape)

Students: (20, 7) | Jobs: (9, 7) | Matches: (180, 6)


## Cell 3 — Core helper functions (reused logic from Task 1-3)

In [3]:
def parse_skills(skill_str):
    """Convert 'Python:85,SQL:75' -> {'Python': 85, 'SQL': 75}"""
    result = {}
    if pd.isna(skill_str) or str(skill_str).strip() == '':
        return result
    for part in str(skill_str).split(','):
        part = part.strip()
        if ':' in part:
            skill, score = part.split(':', 1)
            try:
                result[skill.strip()] = int(score.strip())
            except ValueError:
                pass
    return result


def compute_skill_overlap(student_skills_str, job_skills_str):
    student_skills = parse_skills(student_skills_str)
    required_skills = parse_skills(job_skills_str)
    if not required_skills:
        return 0, 0.0
    met = sum(1 for skill, req in required_skills.items() if student_skills.get(skill, 0) >= req)
    return met, round(met / len(required_skills), 3)


def compute_experience_gap(internship_months, min_exp_years):
    student_exp_years = internship_months / 6.0
    return round(student_exp_years - min_exp_years, 2)


def validate_thresholds(student_id, job_id):
    """Per-skill pass/fail check. Returns (dict_of_results, overall_passed)."""
    student = students[students['student_id'] == student_id].iloc[0]
    job     = jobs[jobs['job_id'] == job_id].iloc[0]

    student_skills  = parse_skills(student['skills'])
    required_skills = parse_skills(job['required_skills'])

    results = {}
    for skill, required_level in required_skills.items():
        student_level = student_skills.get(skill, 0)
        results[skill] = {
            'required_level': required_level,
            'student_level': student_level,
            'passed': bool(student_level >= required_level)
        }

    overall_passed = all(r['passed'] for r in results.values()) if results else False
    return results, overall_passed


print('Helper functions ready')

Helper functions ready


## Cell 4 — Explainability Payload Schema

This is the core deliverable: a structured JSON object attached to every match,
not just a printed demo. Frontend/company views consume this directly.

### Payload fields:
- `student_id`, `job_id` — identifiers
- `match_score` — overall fit (0.0-1.0), based on skill overlap ratio
- `decision` — 'MATCH' or 'NO_MATCH'
- `skill_breakdown` — list of per-skill {skill, required, student_has, passed}
- `experience_check` — {required_years, student_years, gap, passed}
- `reason` — plain-English explanation string
- `missing_skills` — list of skills the student needs to improve
- `edge_case_flags` — list of any data issues encountered (empty list = none)

In [4]:
def build_explanation_payload(student_id, job_id):
    """
    Builds a structured explanation payload for one student-job pair.
    Handles edge cases gracefully: missing student/job, zero required skills,
    zero overlap, missing experience data.
    """
    edge_case_flags = []

    # --- Edge case: student or job not found ---
    student_match = students[students['student_id'] == student_id]
    job_match = jobs[jobs['job_id'] == job_id]

    if student_match.empty:
        return {
            'student_id': int(student_id), 'job_id': int(job_id), 'match_score': 0.0,
            'decision': 'ERROR', 'skill_breakdown': [], 'experience_check': {},
            'reason': f'Student {student_id} not found in records.',
            'missing_skills': [], 'edge_case_flags': ['student_not_found']
        }
    if job_match.empty:
        return {
            'student_id': int(student_id), 'job_id': int(job_id), 'match_score': 0.0,
            'decision': 'ERROR', 'skill_breakdown': [], 'experience_check': {},
            'reason': f'Job {job_id} not found in records.',
            'missing_skills': [], 'edge_case_flags': ['job_not_found']
        }

    student = student_match.iloc[0]
    job = job_match.iloc[0]

    # --- Edge case: job has no required skills listed ---
    required_skills = parse_skills(job['required_skills'])
    if not required_skills:
        edge_case_flags.append('job_has_no_required_skills')

    student_skills = parse_skills(student['skills'])
    if not student_skills:
        edge_case_flags.append('student_has_no_listed_skills')

    # --- Skill breakdown ---
    threshold_results, thresholds_passed = validate_thresholds(student_id, job_id)
    skill_breakdown = [
        {
            'skill': skill,
            'required': r['required_level'],
            'student_has': r['student_level'],
            'passed': r['passed']
        }
        for skill, r in threshold_results.items()
    ]

    missing_skills = [s['skill'] for s in skill_breakdown if not s['passed']]

    # --- Overlap ratio / match score ---
    overlap_count, overlap_ratio = compute_skill_overlap(student['skills'], job['required_skills'])

    # --- Experience check ---
    exp_gap = compute_experience_gap(student['internship_months'], job['min_experience_years'])
    experience_check = {
        'required_years': job['min_experience_years'],
        'student_years': round(student['internship_months'] / 6.0, 2),
        'gap': exp_gap,
        'passed': bool(exp_gap >= -0.5)
    }

    decision = 'MATCH' if (overlap_ratio >= 0.6 and experience_check['passed']) else 'NO_MATCH'

    # --- Plain-English reason ---
    if decision == 'MATCH':
        reason = (f"Student meets {overlap_count}/{len(required_skills)} required skills "
                   f"({overlap_ratio*100:.0f}%) and experience is sufficient "
                   f"({experience_check['student_years']} yrs vs {experience_check['required_years']} yrs required).")
    else:
        reasons = []
        if overlap_ratio < 0.6:
            reasons.append(f"only meets {overlap_count}/{len(required_skills)} required skills "
                            f"({', '.join(missing_skills) if missing_skills else 'none'} missing or below threshold)")
        if not experience_check['passed']:
            reasons.append(f"experience is {abs(exp_gap)} years short of the requirement")
        reason = 'Not a match because: ' + '; '.join(reasons) + '.'

    return {
        'student_id': int(student_id),
        'job_id': int(job_id),
        'match_score': float(round(overlap_ratio, 3)),
        'decision': decision,
        'skill_breakdown': [{k: (int(v) if isinstance(v, (int, np.integer)) else (bool(v) if isinstance(v, (bool, np.bool_)) else v)) for k, v in s.items()} for s in skill_breakdown],
        'experience_check': {k: (float(v) if isinstance(v, (int, float, np.integer, np.floating)) else (bool(v) if isinstance(v, (bool, np.bool_)) else v)) for k, v in experience_check.items()},
        'reason': reason,
        'missing_skills': missing_skills,
        'edge_case_flags': edge_case_flags
    }


print('build_explanation_payload() ready')

build_explanation_payload() ready


## Cell 5 — Live Demo: One Real Example

In [5]:
sample_student_id = students['student_id'].iloc[0]
sample_job_id = jobs['job_id'].iloc[0]

payload = build_explanation_payload(sample_student_id, sample_job_id)
print('=== EXPLANATION PAYLOAD (live example) ===')
print(json.dumps(payload, indent=2))

=== EXPLANATION PAYLOAD (live example) ===
{
  "student_id": 1,
  "job_id": 101,
  "match_score": 1.0,
  "decision": "MATCH",
  "skill_breakdown": [
    {
      "skill": "Python",
      "required": 70,
      "student_has": 85,
      "passed": 1
    },
    {
      "skill": "SQL",
      "required": 60,
      "student_has": 75,
      "passed": 1
    },
    {
      "skill": "Excel",
      "required": 50,
      "student_has": 70,
      "passed": 1
    }
  ],
  "experience_check": {
    "required_years": 1.0,
    "student_years": 3.0,
    "gap": 2.0,
    "passed": 1.0
  },
  "reason": "Student meets 3/3 required skills (100%) and experience is sufficient (3.0 yrs vs 1 yrs required).",
  "missing_skills": [],
  "edge_case_flags": []
}


## Cell 6 — Run at Scale: Generate Payloads for ALL Pairs in matches.csv

Proves this works on real sample data, not just one hardcoded example.

In [6]:
all_payloads = []
for _, row in matches.iterrows():
    payload = build_explanation_payload(row['student_id'], row['job_id'])
    all_payloads.append(payload)

print(f'Generated {len(all_payloads)} explanation payloads')
print()
decisions = pd.Series([p['decision'] for p in all_payloads]).value_counts()
print('Decision breakdown:')
print(decisions)
print()
edge_cases_found = sum(1 for p in all_payloads if p['edge_case_flags'])
print(f'Pairs with edge-case flags: {edge_cases_found}')

Generated 180 explanation payloads

Decision breakdown:
NO_MATCH    158
MATCH        22
Name: count, dtype: int64

Pairs with edge-case flags: 0


## Cell 7 — Validate Explanation Accuracy Against True Labels

Checks whether the `decision` field in our payload actually agrees with the
ground-truth `label` in matches.csv — precision, recall, false-positive rate.

In [7]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix

y_true = matches['label'].values
y_pred = [1 if p['decision'] == 'MATCH' else 0 for p in all_payloads]

print('=== EXPLANATION DECISION ACCURACY (vs true labels) ===')
print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.3f}")
print(f"Recall   : {recall_score(y_true, y_pred, zero_division=0):.3f}")

cm = confusion_matrix(y_true, y_pred)
print(f'Confusion Matrix:\n{cm}')
if cm.size == 4:
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    print(f'False Positive Rate: {fpr:.3f}')

=== EXPLANATION DECISION ACCURACY (vs true labels) ===
Precision: 1.000
Recall   : 1.000
Confusion Matrix:
[[158   0]
 [  0  22]]
False Positive Rate: 0.000


## Cell 8 — Edge Case Tests

Explicitly test the function against bad/missing inputs to prove it doesn't crash.

In [8]:
print('=== EDGE CASE 1: Non-existent student ===')
print(json.dumps(build_explanation_payload(99999, jobs['job_id'].iloc[0]), indent=2))
print()
print('=== EDGE CASE 2: Non-existent job ===')
print(json.dumps(build_explanation_payload(students['student_id'].iloc[0], 99999), indent=2))

=== EDGE CASE 1: Non-existent student ===
{
  "student_id": 99999,
  "job_id": 101,
  "match_score": 0.0,
  "decision": "ERROR",
  "skill_breakdown": [],
  "experience_check": {},
  "reason": "Student 99999 not found in records.",
  "missing_skills": [],
  "edge_case_flags": [
    "student_not_found"
  ]
}

=== EDGE CASE 2: Non-existent job ===
{
  "student_id": 1,
  "job_id": 99999,
  "match_score": 0.0,
  "decision": "ERROR",
  "skill_breakdown": [],
  "experience_check": {},
  "reason": "Job 99999 not found in records.",
  "missing_skills": [],
  "edge_case_flags": [
    "job_not_found"
  ]
}


## Cell 9 — Save All Payloads to JSON (for Frontend hand-off)

In [9]:
with open('explanation_payloads.json', 'w') as f:
    json.dump(all_payloads, f, indent=2)

print(f'Saved {len(all_payloads)} payloads to explanation_payloads.json')
print('This file is the hand-off artifact for Frontend / company views.')

Saved 180 payloads to explanation_payloads.json
This file is the hand-off artifact for Frontend / company views.


## Cell 10 — Payload Schema Documentation (for hand-off)

In [10]:
schema_doc = {
    'student_id'      : 'int — student identifier',
    'job_id'          : 'int — job identifier',
    'match_score'     : 'float (0.0-1.0) — skill overlap ratio',
    'decision'        : 'string — MATCH | NO_MATCH | ERROR',
    'skill_breakdown' : 'list of {skill, required, student_has, passed} — per-skill detail',
    'experience_check': '{required_years, student_years, gap, passed}',
    'reason'          : 'string — plain-English explanation, always present',
    'missing_skills'  : 'list of skill names student is below threshold on',
    'edge_case_flags' : 'list of strings — empty if no data issues, else flags like student_not_found'
}
print('=== PAYLOAD SCHEMA (hand off to Frontend team) ===')
print(json.dumps(schema_doc, indent=2))

=== PAYLOAD SCHEMA (hand off to Frontend team) ===
{
  "student_id": "int \u2014 student identifier",
  "job_id": "int \u2014 job identifier",
  "match_score": "float (0.0-1.0) \u2014 skill overlap ratio",
  "decision": "string \u2014 MATCH | NO_MATCH | ERROR",
  "skill_breakdown": "list of {skill, required, student_has, passed} \u2014 per-skill detail",
  "experience_check": "{required_years, student_years, gap, passed}",
  "reason": "string \u2014 plain-English explanation, always present",
  "missing_skills": "list of skill names student is below threshold on",
  "edge_case_flags": "list of strings \u2014 empty if no data issues, else flags like student_not_found"
}
